In [1]:
import pandas as pd

from libs.SVDModel import SVDModel
from libs.utils import *

In [2]:
subsample_name = 'ur0.01_ir0.01'
content_embedding_size = 64  # [1..64]

train_interactions_files = [f'subsamples/{subsample_name}/train/week_{i:02}.parquet' for i in range(25)]
val_interactions_file = [f'subsamples/{subsample_name}/validation/week_25.parquet']

In [3]:
# download_files(['metadata/users_metadata.parquet', 'metadata/items_metadata.parquet', 'metadata/item_embeddings.npz'])

In [4]:
(train_df, val_df), _, _ = load_data(
    train_interactions_files, val_interactions_file, content_embedding_size
)
print(f"Train: {len(train_df)} Val: {len(val_df)}")

train_df = train_df[train_df[TARGET] != 0]
val_df = val_df[val_df[TARGET] == 1]

print(f"Train: {len(train_df)} Val: {len(val_df)}")

Train: 3720328 Val: 170111
Train: 172235 Val: 8179


In [5]:
model = SVDModel(200).fit(train_df)

In [6]:
predict = model.batch_recommend_users_for_items(
    item_ids=val_df[ITEM].unique().tolist(),
    n_recommendations=100,
    batch_size=1000
)

Predict batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [7]:
submission = pd.DataFrame([(item, users) for item, (users, score) in predict.items()], columns=[ITEM, "user_ids"])
true_positive_reactions = val_df.groupby(ITEM).agg(user_ids=(USER, list)).reset_index()

In [14]:
print(f"Val NDCG: {NDCG(submission, true_positive_reactions):.3f}")

TypeError: "value" parameter must be a scalar or dict, but you passed a "list"